# JD 후보자 매칭 시스템
**사용 방법:** 위에서부터 순서대로 셀을 실행하세요 (Shift+Enter)

---

In [ ]:
# [1단계] 필요한 패키지 설치 (최초 1회만 실행)
!pip install pdfplumber google-generativeai -q
print('✅ 설치 완료')

In [ ]:
# [2단계] Gemini API 키 입력
import google.generativeai as genai
import getpass

api_key = getpass.getpass('Gemini API 키를 입력하세요: ')
genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-2.0-flash')
print('✅ API 키 설정 완료')

In [ ]:
# [3단계] 분석 함수 정의 (그냥 실행만 하세요)
import json, re, io
import pdfplumber

EDUCATION_LEVELS = {'고졸': 1, '전문대졸': 2, '대졸': 3, '대학원졸': 4, '석사': 4, '박사': 5}

def extract_text(file_bytes):
    parts = []
    with pdfplumber.open(io.BytesIO(file_bytes)) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t: parts.append(t)
    return '\n'.join(parts)

def extract_json(text):
    m = re.search(r'```json\s*([\s\S]*?)\s*```', text)
    if m: return json.loads(m.group(1))
    m = re.search(r'\{[\s\S]*\}', text)
    if m: return json.loads(m.group(0))
    raise ValueError('JSON 없음')

def analyze_jd(text):
    prompt = f'''다음 채용공고를 분석하여 JSON으로 추출하세요. 없는 정보는 null.

{text}

반드시 아래 형식으로만 응답:
```json
{{
  "position": "직무명",
  "required_skills": ["스킬1"],
  "preferred_skills": ["스킬1"],
  "min_experience_years": 3,
  "max_experience_years": null,
  "education": "대졸",
  "required_certifications": [],
  "domain": "도메인",
  "key_responsibilities": ["업무1"],
  "job_description": "한 줄 요약"
}}
```'''
    r = model.generate_content(prompt)
    return extract_json(r.text)

def analyze_resume(text, filename):
    prompt = f'''다음 이력서를 분석하여 JSON으로 추출하세요. 없는 정보는 null. 경력은 소수 허용, 연봉은 만원 단위 정수.

{text}

반드시 아래 형식으로만 응답:
```json
{{
  "name": "홍길동",
  "age": 30,
  "education": "대졸",
  "education_major": "컴퓨터공학",
  "total_experience_years": 5.0,
  "skills": ["Python"],
  "certifications": [],
  "last_salary": 4500,
  "companies": ["회사명"],
  "projects": ["프로젝트"],
  "career_summary": "경력 요약"
}}
```'''
    try:
        r = model.generate_content(prompt)
        d = extract_json(r.text)
        d['filename'] = filename
        return d
    except Exception as e:
        return {'filename': filename, 'name': filename, 'parse_error': str(e),
                'skills': [], 'certifications': [], 'companies': [], 'projects': []}

def apply_filter(profile, criteria):
    reasons = []
    exp = profile.get('total_experience_years')
    if criteria.get('min_experience_years') and (exp is None or exp < criteria['min_experience_years']):
        reasons.append(f'경력 부족 ({exp}년 < {criteria["min_experience_years"]}년)')
    age = profile.get('age')
    if criteria.get('min_age') and (age is None or age < criteria['min_age']):
        reasons.append(f'나이 미달 ({age}세 < {criteria["min_age"]}세)')
    if criteria.get('max_age') and (age is None or age > criteria['max_age']):
        reasons.append(f'나이 초과 ({age}세 > {criteria["max_age"]}세)')
    edu = profile.get('education')
    min_edu = criteria.get('min_education')
    if min_edu and (edu is None or EDUCATION_LEVELS.get(edu, 0) < EDUCATION_LEVELS.get(min_edu, 0)):
        reasons.append(f'학력 미달 ({edu} < {min_edu})')
    salary = profile.get('last_salary')
    if criteria.get('min_last_salary') and (salary is None or salary < criteria['min_last_salary']):
        reasons.append(f'연봉 미달 ({salary}만원 < {criteria["min_last_salary"]}만원)')
    for cert in criteria.get('required_certifications', []):
        if cert and cert not in (profile.get('certifications') or []):
            reasons.append(f'자격증 없음: {cert}')
    return len(reasons) == 0, reasons

def match_candidate(jd, candidate):
    jd_summary = f'''직무: {jd.get("position")}
도메인: {jd.get("domain")}
필수 스킬: {", ".join(jd.get("required_skills") or [])}
우대 스킬: {", ".join(jd.get("preferred_skills") or [])}
최소 경력: {jd.get("min_experience_years")}년 이상
요구 학력: {jd.get("education") or "무관"}
주요 업무: {", ".join((jd.get("key_responsibilities") or [])[:5])}'''

    cand_summary = f'''이름: {candidate.get("name")}
나이: {candidate.get("age") or "미확인"}세
학력: {candidate.get("education") or "미확인"} ({candidate.get("education_major")})
경력: {candidate.get("total_experience_years") or "미확인"}년
스킬: {", ".join(candidate.get("skills") or [])}
자격증: {", ".join(candidate.get("certifications") or []) or "없음"}
경력 요약: {candidate.get("career_summary")}'''

    prompt = f'''채용 전문가로서 아래 채용공고와 후보자를 비교하여 평가하세요.

=== 채용공고 ===
{jd_summary}

=== 후보자 ===
{cand_summary}

반드시 아래 JSON으로만 응답:
```json
{{
  "total_score": 85,
  "skill_match_score": 90,
  "experience_score": 80,
  "education_score": 100,
  "overall_fit": "상",
  "strengths": ["강점1", "강점2"],
  "weaknesses": ["약점1"],
  "recommendation": "적극 추천",
  "reasoning": "종합 평가 3~5문장"
}}
```
overall_fit: "상"/"중"/"하" 중 하나
recommendation: "적극 추천"/"추천"/"검토 필요"/"미추천" 중 하나
total_score = 스킬 40% + 경력 35% + 학력 15% + 기타 10%'''

    r = model.generate_content(prompt)
    return extract_json(r.text)

print('✅ 함수 정의 완료')

In [ ]:
# [4단계] JD(채용공고) PDF 업로드
from google.colab import files

print('📄 채용공고(JD) PDF 파일을 선택하세요')
jd_uploaded = files.upload()
jd_filename = list(jd_uploaded.keys())[0]
jd_bytes = jd_uploaded[jd_filename]
print(f'✅ 업로드 완료: {jd_filename}')

In [ ]:
# [5단계] 이력서 PDF 업로드 (여러 개 동시 선택 가능)
print('📄 이력서 PDF 파일들을 선택하세요 (여러 개 동시 선택 가능)')
resume_uploaded = files.upload()
print(f'✅ {len(resume_uploaded)}개 이력서 업로드 완료')
for name in resume_uploaded.keys():
    print(f'  - {name}')

In [ ]:
# [6단계] 필터 조건 설정 (필요 없으면 None 그대로 두세요)

filter_criteria = {
    'min_experience_years': None,   # 최소 경력(년). 예: 3
    'min_age': None,                # 최소 나이. 예: 25
    'max_age': None,                # 최대 나이. 예: 45
    'min_education': None,          # 최소 학력. '고졸'/'전문대졸'/'대졸'/'대학원졸' 중 하나
    'min_last_salary': None,        # 최소 연봉(만원). 예: 3000
    'required_certifications': [],  # 필수 자격증. 예: ['정보처리기사', 'SQLD']
}

print('✅ 필터 조건 설정 완료')
print(filter_criteria)

In [ ]:
# [7단계] 분석 실행

print('🔍 JD 분석 중...')
jd_text = extract_text(jd_bytes)
jd = analyze_jd(jd_text)
print(f'✅ JD 분석 완료: {jd.get("position")} / {jd.get("domain")}')

results = []
filtered_out = []

total = len(resume_uploaded)
for i, (filename, file_bytes) in enumerate(resume_uploaded.items()):
    print(f'\n[{i+1}/{total}] {filename} 분석 중...')

    resume_text = extract_text(file_bytes)
    profile = analyze_resume(resume_text, filename)

    passed, reasons = apply_filter(profile, filter_criteria)

    if not passed:
        print(f'  ❌ 필터 탈락: {", ".join(reasons)}')
        filtered_out.append({'profile': profile, 'reasons': reasons})
    else:
        print(f'  ✅ 필터 통과 → 매칭 평가 중...')
        try:
            match = match_candidate(jd, profile)
            match['filename'] = filename
            match['profile'] = profile
            results.append(match)
            print(f'  🎯 점수: {match.get("total_score")}점 ({match.get("recommendation")})')
        except Exception as e:
            print(f'  ⚠️ 매칭 오류: {e}')

results.sort(key=lambda x: x.get('total_score', 0), reverse=True)
print(f'\n🏁 분석 완료! 통과: {len(results)}명 / 탈락: {len(filtered_out)}명')

In [ ]:
# [8단계] 결과 시각화
from IPython.display import display, HTML

def score_bar(score):
    color = '#22c55e' if score >= 80 else '#eab308' if score >= 60 else '#ef4444'
    return f'''<div style="font-size:11px;color:#64748b;margin-bottom:2px">{score}점</div>
    <div style="background:#e2e8f0;border-radius:4px;height:6px;width:100%">
      <div style="background:{color};height:6px;border-radius:4px;width:{score}%"></div>
    </div>'''

html = f'''
<style>
  .card{{background:#fff;border:1px solid #e2e8f0;border-radius:12px;padding:16px;margin-bottom:12px;font-family:sans-serif}}
  .badge{{display:inline-block;padding:2px 8px;border-radius:20px;font-size:11px;font-weight:600;margin-left:4px}}
  .fit-상{{background:#d1fae5;color:#065f46}} .fit-중{{background:#fef3c7;color:#92400e}} .fit-하{{background:#fee2e2;color:#991b1b}}
  .rec-적극추천{{background:#dbeafe;color:#1e40af}} .rec-추천{{background:#d1fae5;color:#065f46}}
  .rec-검토필요{{background:#fef9c3;color:#92400e}} .rec-미추천{{background:#fee2e2;color:#991b1b}}
</style>

<h2 style="font-family:sans-serif;color:#4338ca">분석 결과</h2>
<p style="font-family:sans-serif;color:#64748b">직무: <b>{jd.get("position")}</b> | 도메인: <b>{jd.get("domain")}</b></p>

<div style="display:flex;gap:12px;margin-bottom:16px">
  <div class="card" style="text-align:center;min-width:100px">
    <div style="font-size:28px;font-weight:bold;color:#4338ca">{total}</div>
    <div style="font-size:12px;color:#94a3b8">총 이력서</div>
  </div>
  <div class="card" style="text-align:center;min-width:100px">
    <div style="font-size:28px;font-weight:bold;color:#ef4444">{len(filtered_out)}</div>
    <div style="font-size:12px;color:#94a3b8">필터 탈락</div>
  </div>
  <div class="card" style="text-align:center;min-width:100px">
    <div style="font-size:28px;font-weight:bold;color:#22c55e">{len(results)}</div>
    <div style="font-size:12px;color:#94a3b8">분석 완료</div>
  </div>
  <div class="card" style="text-align:center;min-width:100px">
    <div style="font-size:28px;font-weight:bold;color:#3b82f6">{len([r for r in results if r.get("total_score", 0) >= 70])}</div>
    <div style="font-size:12px;color:#94a3b8">상위 후보</div>
  </div>
</div>
'''

if filtered_out:
    html += '<div class="card"><b style="color:#64748b">필터 탈락 후보자</b><br>'
    for f in filtered_out:
        name = f['profile'].get('name') or f['profile'].get('filename')
        html += f'<div style="margin-top:6px;font-size:13px">❌ <b>{name}</b> — {", ".join(f["reasons"])}</div>'
    html += '</div>'

html += '<h3 style="font-family:sans-serif;color:#1e293b">매칭 결과 (점수 순위)</h3>'

for i, r in enumerate(results):
    p = r.get('profile', {})
    score = r.get('total_score', 0)
    score_color = '#16a34a' if score >= 80 else '#ca8a04' if score >= 60 else '#dc2626'
    fit = r.get('overall_fit', '')
    rec = r.get('recommendation', '').replace(' ', '')
    name = p.get('name') or p.get('filename', '')

    skills_html = ''.join([f'<span style="background:#f1f5f9;color:#475569;font-size:11px;padding:2px 8px;border-radius:20px;margin:2px;display:inline-block">{s}</span>' for s in (p.get('skills') or [])])
    strengths_html = ''.join([f'<li style="color:#16a34a;font-size:13px">✓ {s}</li>' for s in (r.get('strengths') or [])])
    weaknesses_html = ''.join([f'<li style="color:#dc2626;font-size:13px">✗ {s}</li>' for s in (r.get('weaknesses') or [])])

    html += f'''
    <div class="card">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:12px">
        <div style="display:flex;align-items:center;gap:10px">
          <div style="width:32px;height:32px;border-radius:50%;background:#e0e7ff;color:#4338ca;display:flex;align-items:center;justify-content:center;font-weight:bold">{i+1}</div>
          <div>
            <div style="font-weight:600;font-size:15px">{name}</div>
            <div style="font-size:11px;color:#94a3b8">{p.get("filename", "")}</div>
          </div>
        </div>
        <div style="text-align:right">
          <span style="font-size:28px;font-weight:bold;color:{score_color}">{score}</span><span style="color:#94a3b8">점</span>
          <div><span class="badge fit-{fit}">{fit}급</span><span class="badge rec-{rec}">{r.get("recommendation")}</span></div>
        </div>
      </div>

      <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:12px;margin-bottom:12px;padding:10px;background:#f8fafc;border-radius:8px">
        <div><div style="font-size:11px;color:#94a3b8">스킬 매칭</div>{score_bar(r.get("skill_match_score", 0))}</div>
        <div><div style="font-size:11px;color:#94a3b8">경력 적합</div>{score_bar(r.get("experience_score", 0))}</div>
        <div><div style="font-size:11px;color:#94a3b8">학력 충족</div>{score_bar(r.get("education_score", 0))}</div>
      </div>

      <div style="font-size:12px;color:#64748b;margin-bottom:8px">
        {'<span style="margin-right:12px">나이: <b>' + str(p.get('age')) + '세</b></span>' if p.get('age') else ''}
        {'<span style="margin-right:12px">경력: <b>' + str(p.get('total_experience_years')) + '년</b></span>' if p.get('total_experience_years') is not None else ''}
        {'<span style="margin-right:12px">학력: <b>' + str(p.get('education')) + '</b></span>' if p.get('education') else ''}
        {'<span>연봉: <b>' + str(p.get('last_salary')) + '만원</b></span>' if p.get('last_salary') else ''}
      </div>

      {'<div style="margin-bottom:8px">' + skills_html + '</div>' if skills_html else ''}

      <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:8px">
        <div><div style="font-size:11px;color:#94a3b8;margin-bottom:4px">강점</div><ul style="margin:0;padding-left:0;list-style:none">{strengths_html}</ul></div>
        <div><div style="font-size:11px;color:#94a3b8;margin-bottom:4px">약점</div><ul style="margin:0;padding-left:0;list-style:none">{weaknesses_html}</ul></div>
      </div>

      <div style="background:#f8fafc;border-radius:8px;padding:10px">
        <div style="font-size:11px;color:#94a3b8;margin-bottom:4px">AI 평가 의견</div>
        <div style="font-size:13px;color:#475569;line-height:1.6">{r.get("reasoning", "")}</div>
      </div>
    </div>'''

if not results:
    html += '<p style="color:#94a3b8;text-align:center">필터 통과 후보자가 없습니다.</p>'

display(HTML(html))